# 🤖 Matches ML Pipeline
**Target:** `label_relevant` — هل الـ match مناسب للمستخدم؟

---

## 📥 Step 1 — Load Data

In [ ]:
# Step 1: Load Data
import pandas as pd
import numpy as np

# ⬇️  Upload the file manually in Colab OR mount Drive and adjust the path
# from google.colab import files
# uploaded = files.upload()  # uncomment to upload manually

FILE_PATH = 'matches_all_20260515_211601.csv'  # adjust if needed

df = pd.read_csv(FILE_PATH)

print(f'✅ Loaded {df.shape[0]:,} rows × {df.shape[1]} columns')
print()
print('Columns:', list(df.columns))
print()
df.head()

---
## 🧹 Step 2 — Data Cleaning & Preprocessing

In [ ]:
# Step 2: Data Cleaning & Preprocessing

df_clean = df.copy()

# ── 2.1  Drop columns not useful for modeling ──────────────────────────────
drop_cols = [
    'id',               # unique identifier — no signal
    'feedback_type',    # ~44% missing; info already captured in label_relevant
    'label_not_relevant'  # perfectly inverse of label_relevant → data leakage
]
df_clean.drop(columns=drop_cols, inplace=True)
print(f'Dropped columns: {drop_cols}')

# ── 2.2  Encode categorical column ─────────────────────────────────────────
df_clean['match_type'] = df_clean['match_type'].fillna('unknown')
df_clean['match_type'] = df_clean['match_type'].map(
    {'collaborator': 0, 'project': 1, 'unknown': -1}
)
print('Encoded match_type:', df_clean['match_type'].value_counts().to_dict())

# ── 2.3  Fill missing numeric values with column median ────────────────────
num_cols_with_na = df_clean.columns[df_clean.isnull().any()].tolist()
print(f'Columns with NaN before fill: {num_cols_with_na}')

for col in num_cols_with_na:
    median_val = df_clean[col].median()
    df_clean[col].fillna(median_val, inplace=True)

print(f'NaN after fill: {df_clean.isnull().sum().sum()}')

# ── 2.4  Summary ───────────────────────────────────────────────────────────
print(f'\n✅ Clean shape: {df_clean.shape}')
print('Target distribution:')
print(df_clean['label_relevant'].value_counts())
df_clean.head()

---
## ⚙️ Step 3 — Feature Engineering

In [ ]:
# Step 3: Feature Engineering

df_feat = df_clean.copy()

# ── 3.1  Interaction features ──────────────────────────────────────────────
# نسبة المهارات المشتركة من إجمالي المهارات
df_feat['total_skills'] = df_feat['skills_count_a'] + df_feat['skills_count_b']
df_feat['overlap_ratio'] = np.where(
    df_feat['total_skills'] > 0,
    df_feat['overlapping_skills_count'] / df_feat['total_skills'],
    0
)

# مدى تفاعل المستخدم (viewed + saved + action)
df_feat['engagement_score'] = df_feat['viewed'] + df_feat['saved'] + df_feat['action_taken']

# هل الطرفان verified؟
df_feat['trust_score'] = df_feat['user_a_verified'].fillna(0) + df_feat['user_b_verified'].fillna(0)

# نسبة التكامل بين المهارات
df_feat['skill_gap'] = abs(df_feat['skills_count_a'] - df_feat['skills_count_b'])

# ── 3.2  Composite readiness ───────────────────────────────────────────────
# يجمع project_accepting + team_openness كمؤشر جاهزية للتعاون
df_feat['readiness'] = df_feat['project_accepting'] + df_feat['team_openness']

# ── 3.3  Location × identity signal ───────────────────────────────────────
df_feat['local_verified'] = df_feat['same_location'] * df_feat['user_identity_verified']

print('✅ New features added:')
new_feats = ['total_skills', 'overlap_ratio', 'engagement_score', 'trust_score',
             'skill_gap', 'readiness', 'local_verified']
print(df_feat[new_feats].describe().round(3))
print(f'\nFinal feature count: {df_feat.shape[1]} columns')

---
## 🏋️ Step 4 — Train Model

In [ ]:
# Step 4: Train Model
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ── 4.1  Define features & target ─────────────────────────────────────────
TARGET = 'label_relevant'
EXCLUDE = [TARGET]

X = df_feat.drop(columns=EXCLUDE)
y = df_feat[TARGET]

# ── 4.2  Train / test split (stratified because class imbalance) ──────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows')
print(f'Positive rate — Train: {y_train.mean():.2%} | Test: {y_test.mean():.2%}')

# ── 4.3  Scale ─────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── 4.4  Train Random Forest (handles imbalance with class_weight) ─────────
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    class_weight='balanced',  # مهم بسبب الـ imbalance (87% / 13%)
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_sc, y_train)

# ── 4.5  Cross-validation ─────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(rf, X_train_sc, y_train, cv=cv, scoring='roc_auc')
print(f'\n5-Fold CV ROC-AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')

# ── 4.6  Feature importance plot ──────────────────────────────────────────
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=feat_imp.values[:15], y=feat_imp.index[:15], palette='viridis')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print('\n✅ Model trained successfully!')

---
## 🔮 Step 5 — Predictions & Evaluation

In [ ]:
# Step 5: Predictions & Evaluation

# ── 5.1  Predict ──────────────────────────────────────────────────────────
y_pred      = rf.predict(X_test_sc)
y_pred_prob = rf.predict_proba(X_test_sc)[:, 1]  # احتمال أن الـ match relevant

# ── 5.2  Metrics ──────────────────────────────────────────────────────────
print('=' * 55)
print('Classification Report')
print('=' * 55)
print(classification_report(y_test, y_pred, target_names=['Not Relevant', 'Relevant']))

auc = roc_auc_score(y_test, y_pred_prob)
print(f'ROC-AUC Score: {auc:.4f}')

# ── 5.3  Confusion Matrix ─────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Relevant', 'Relevant'],
            yticklabels=['Not Relevant', 'Relevant'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# ── 5.4  ROC Curve ────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'RF AUC = {auc:.4f}', color='steelblue', lw=2)
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

print('\n✅ Predictions done!')

---
## 💾 Step 6 — Export Results

In [ ]:
# Step 6: Export Results
import joblib

# ── 6.1  Add predictions to test set and save CSV ─────────────────────────
results_df = X_test.copy()
results_df['actual_label']       = y_test.values
results_df['predicted_label']    = y_pred
results_df['predicted_prob']     = y_pred_prob.round(4)
results_df['correct_prediction'] = (results_df['actual_label'] == results_df['predicted_label']).astype(int)

results_csv = 'matches_predictions.csv'
results_df.to_csv(results_csv, index=False)
print(f'✅ Predictions saved → {results_csv}')

# ── 6.2  Save model & scaler ──────────────────────────────────────────────
joblib.dump(rf,     'rf_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print('✅ Model saved  → rf_model.pkl')
print('✅ Scaler saved → scaler.pkl')

# ── 6.3  Summary stats ────────────────────────────────────────────────────
print('\n' + '='*45)
print('FINAL SUMMARY')
print('='*45)
print(f'Total test samples :  {len(y_test)}')
print(f'Correct predictions:  {results_df["correct_prediction"].sum()}')
print(f'Accuracy           :  {results_df["correct_prediction"].mean():.2%}')
print(f'ROC-AUC            :  {auc:.4f}')
print('='*45)

# ── 6.4  Download in Colab ────────────────────────────────────────────────
try:
    from google.colab import files
    files.download(results_csv)
    files.download('rf_model.pkl')
    print('\n📥 Files downloaded to your machine!')
except ImportError:
    print('\n(Not in Colab — find the files in your working directory)')